# Phase 1: Data Understanding and Descriptive Statistics

## 1. Project Setup and Data Loading

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# مسیر اصلی پروژه
PROJECT_DIR = Path.cwd().parent

# مسیر فایل‌های خام
RAW_DIR = PROJECT_DIR / "data" / "raw"

print("Project directory:", PROJECT_DIR)
print("Raw data directory:", RAW_DIR)

# نمایش فایل‌های موجود
for file in RAW_DIR.iterdir():
    print(file.name, "|", file.stat().st_size, "bytes")

Project directory: c:\Users\abaas\OneDrive\Desktop\آمار و کاربرد آن\NHANES_Project
Raw data directory: c:\Users\abaas\OneDrive\Desktop\آمار و کاربرد آن\NHANES_Project\data\raw
BMX_L.xpt | 1563200 bytes
BPXO_L.xpt | 696720 bytes
DEMO_L.xpt | 2582160 bytes


## 2. Initial Data Exploration

In [3]:
# خواندن فایل‌های NHANES
demo = pd.read_sas(RAW_DIR / "DEMO_L.xpt")
bmx = pd.read_sas(RAW_DIR / "BMX_L.xpt")
bpxo = pd.read_sas(RAW_DIR / "BPXO_L.xpt")

# نمایش ابعاد فایل‌ها
print("DEMO_L:", demo.shape)
print("BMX_L:", bmx.shape)
print("BPXO_L:", bpxo.shape)

DEMO_L: (11933, 27)
BMX_L: (8860, 22)
BPXO_L: (7801, 12)


In [4]:
print("DEMO_L variables:")
print(demo.columns.tolist())

print("\nBMX_L variables:")
print(bmx.columns.tolist())

print("\nBPXO_L variables:")
print(bpxo.columns.tolist())

DEMO_L variables:
['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'INDFMPIR']

BMX_L variables:
['SEQN', 'BMDSTATS', 'BMXWT', 'BMIWT', 'BMXRECUM', 'BMIRECUM', 'BMXHEAD', 'BMIHEAD', 'BMXHT', 'BMIHT', 'BMXBMI', 'BMDBMIC', 'BMXLEG', 'BMILEG', 'BMXARML', 'BMIARML', 'BMXARMC', 'BMIARMC', 'BMXWAIST', 'BMIWAIST', 'BMXHIP', 'BMIHIP']

BPXO_L variables:
['SEQN', 'BPAOARM', 'BPAOCSZ', 'BPXOSY1', 'BPXODI1', 'BPXOSY2', 'BPXODI2', 'BPXOSY3', 'BPXODI3', 'BPXOPLS1', 'BPXOPLS2', 'BPXOPLS3']


## 3. Data Quality Assessment

In [5]:
# بررسی شناسه‌های شرکت‌کنندگان در هر فایل

datasets = {
    "DEMO_L": demo,
    "BMX_L": bmx,
    "BPXO_L": bpxo
}

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 30)

    print("Total rows:", len(df))
    print("Unique SEQN:", df["SEQN"].nunique())
    print("Missing SEQN:", df["SEQN"].isna().sum())
    print("Duplicate SEQN:", df["SEQN"].duplicated().sum())


DEMO_L
------------------------------
Total rows: 11933
Unique SEQN: 11933
Missing SEQN: 0
Duplicate SEQN: 0

BMX_L
------------------------------
Total rows: 8860
Unique SEQN: 8860
Missing SEQN: 0
Duplicate SEQN: 0

BPXO_L
------------------------------
Total rows: 7801
Unique SEQN: 7801
Missing SEQN: 0
Duplicate SEQN: 0


In [6]:
# تعداد اندازه‌گیری‌های معتبر فشار سیستولیک برای هر فرد

bp_columns = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]

bpxo["Valid_Systolic_Readings"] = bpxo[bp_columns].notna().sum(axis=1)

print(bpxo["Valid_Systolic_Readings"].value_counts().sort_index())

Valid_Systolic_Readings
0     283
1      12
2      28
3    7478
Name: count, dtype: int64


In [7]:
# متغیرهای اصلی موردنیاز پروژه
key_variables = {
    "DEMO_L": ["RIDAGEYR", "RIAGENDR", "WTMEC2YR"],
    "BMX_L": ["BMXBMI"],
    "BPXO_L": ["BPXOSY1", "BPXOSY2", "BPXOSY3"]
}

# بررسی مقادیر گمشده
for name, variables in key_variables.items():
    df = datasets[name]

    print(f"\n{name}")
    print("-" * 40)

    for var in variables:
        missing = df[var].isna().sum()
        percentage = missing / len(df) * 100

        print(
            f"{var}: {missing} missing "
            f"({percentage:.2f}%)"
        )


DEMO_L
----------------------------------------
RIDAGEYR: 0 missing (0.00%)
RIAGENDR: 0 missing (0.00%)
WTMEC2YR: 0 missing (0.00%)

BMX_L
----------------------------------------
BMXBMI: 389 missing (4.39%)

BPXO_L
----------------------------------------
BPXOSY1: 284 missing (3.64%)
BPXOSY2: 296 missing (3.79%)
BPXOSY3: 321 missing (4.11%)


## 4. Age Cleaning and Adult Population Selection

In [8]:
# بررسی وضعیت شرکت‌کنندگان
print("RIDSTATR - Examination Status:")
print(demo["RIDSTATR"].value_counts(dropna=False).sort_index())

# بررسی توزیع سن
print("\nAge Statistics:")
print(demo["RIDAGEYR"].describe())

# تعداد افراد زیر و بالای 20 سال
print("\nAge Groups:")
print("Under 20:", (demo["RIDAGEYR"] < 20).sum())
print("20 and older:", (demo["RIDAGEYR"] >= 20).sum())

RIDSTATR - Examination Status:
RIDSTATR
1.0    3073
2.0    8860
Name: count, dtype: int64

Age Statistics:
count    1.193300e+04
mean     3.831786e+01
std      2.560199e+01
min      5.397605e-79
25%      1.300000e+01
50%      3.700000e+01
75%      6.200000e+01
max      8.000000e+01
Name: RIDAGEYR, dtype: float64

Age Groups:
Under 20: 4124
20 and older: 7809


In [ ]:
print("\nFirst 5 rows:")
display(demo.head())

print("\nSelected columns:")
display(demo[["SEQN", "RIDAGEYR", "RIDSTATR", "RIAGENDR"]].head(10))

['SEQN', 'SDDSRVYR', 'RIDSTATR', 'RIAGENDR', 'RIDAGEYR', 'RIDAGEMN', 'RIDRETH1', 'RIDRETH3', 'RIDEXMON', 'RIDEXAGM', 'DMQMILIZ', 'DMDBORN4', 'DMDYRUSR', 'DMDEDUC2', 'DMDMARTZ', 'RIDEXPRG', 'DMDHHSIZ', 'DMDHRGND', 'DMDHRAGZ', 'DMDHREDZ', 'DMDHRMAZ', 'DMDHSEDZ', 'WTINT2YR', 'WTMEC2YR', 'SDMVSTRA', 'SDMVPSU', 'INDFMPIR']

First 5 rows:


,SEQN,SDDSRVYR,RIDSTATR,RIAGENDR,RIDAGEYR,RIDAGEMN,RIDRETH1,RIDRETH3,RIDEXMON,RIDEXAGM,...,DMDHRGND,DMDHRAGZ,DMDHREDZ,DMDHRMAZ,DMDHSEDZ,WTINT2YR,WTMEC2YR,SDMVSTRA,SDMVPSU,INDFMPIR
0,130378.0,12.0,2.0,1.0,43.0,NaN,5.0,6.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,50055.450807,54374.463898,173.0,2.0,5.00
1,130379.0,12.0,2.0,1.0,66.0,NaN,3.0,3.0,2.0,NaN,...,NaN,NaN,NaN,NaN,NaN,29087.450605,34084.721548,173.0,2.0,5.00
2,130380.0,12.0,2.0,2.0,44.0,NaN,2.0,2.0,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,80062.674301,81196.277992,174.0,1.0,1.41
3,130381.0,12.0,2.0,2.0,5.0,NaN,5.0,7.0,1.0,71.0,...,2.0,2.0,2.0,3.0,NaN,38807.268902,55698.607106,182.0,2.0,1.53
4,130382.0,12.0,2.0,1.0,2.0,NaN,3.0,3.0,2.0,34.0,...,2.0,2.0,3.0,1.0,2.0,30607.519774,36434.146346,182.0,2.0,3.60



Selected columns:


,SEQN,RIDAGEYR,RIDSTATR,RIAGENDR
0,130378.0,43.0,2.0,1.0
1,130379.0,66.0,2.0,1.0
2,130380.0,44.0,2.0,2.0
3,130381.0,5.0,2.0,2.0
4,130382.0,2.0,2.0,1.0
5,130383.0,3.0,1.0,2.0
6,130384.0,43.0,1.0,1.0
7,130385.0,65.0,1.0,2.0
8,130386.0,34.0,2.0,1.0
9,130387.0,68.0,2.0,2.0


In [12]:
# بررسی نوع داده و مقادیر گمشده سن
print("Data type:", demo["RIDAGEYR"].dtype)
print("Missing values:", demo["RIDAGEYR"].isna().sum())

# بررسی کوچک‌ترین و بزرگ‌ترین مقادیر
print("Minimum age:", demo["RIDAGEYR"].min())
print("Maximum age:", demo["RIDAGEYR"].max())

# نمایش فراوانی سن‌های کمتر از 1
print("\nValues below 1:")
print(demo.loc[demo["RIDAGEYR"] < 1, "RIDAGEYR"].value_counts())

Data type: float64
Missing values: 0
Minimum age: 5.397605346934028e-79
Maximum age: 80.0

Values below 1:
RIDAGEYR
5.397605e-79    189
Name: count, dtype: int64


In [13]:
# ایجاد نسخه‌ای از داده‌ها برای پاک‌سازی
demo_clean = demo.copy()

# شناسایی کد غیرعادی در متغیر سن
special_missing = demo_clean["RIDAGEYR"] < 1e-10

# تبدیل این مقادیر به NaN
demo_clean.loc[special_missing, "RIDAGEYR"] = np.nan

# بررسی نتیجه
print("Corrected values:", special_missing.sum())
print("Missing ages after correction:", demo_clean["RIDAGEYR"].isna().sum())
print("Minimum age after correction:", demo_clean["RIDAGEYR"].min())

Corrected values: 189
Missing ages after correction: 189
Minimum age after correction: 1.0


In [14]:
print("Total participants:", len(demo_clean))

print("Missing ages:", demo_clean["RIDAGEYR"].isna().sum())

print("Participants aged 20 or older:",
      (demo_clean["RIDAGEYR"] >= 20).sum())

print("Participants younger than 20:",
      (demo_clean["RIDAGEYR"] < 20).sum())

Total participants: 11933
Missing ages: 189
Participants aged 20 or older: 7809
Participants younger than 20: 3935


## 5. BMI Assessment

In [15]:
# بررسی متغیر BMI در فایل BMX_L

print("Total participants:", len(bmx))
print("Missing BMI values:", bmx["BMXBMI"].isna().sum())

print("\nBMI Statistics:")
print(bmx["BMXBMI"].describe())

print("\nSmallest BMI values:")
print(bmx["BMXBMI"].nsmallest(10).to_list())

print("\nLargest BMI values:")
print(bmx["BMXBMI"].nlargest(10).to_list())

Total participants: 8860
Missing BMI values: 389

BMI Statistics:
count    8471.000000
mean       27.246665
std         8.137781
min        11.100000
25%        21.600000
50%        26.400000
75%        31.700000
max        74.800000
Name: BMXBMI, dtype: float64

Smallest BMI values:
[11.1, 12.4, 12.5, 12.5, 12.7, 12.9, 12.9, 12.9, 13.0, 13.1]

Largest BMI values:
[74.8, 69.9, 69.1, 68.9, 68.5, 67.8, 67.1, 66.6, 66.5, 66.1]


In [16]:
Q1 = bmx["BMXBMI"].quantile(0.25)
Q3 = bmx["BMXBMI"].quantile(0.75)

IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = bmx[
    (bmx["BMXBMI"] < lower_bound) |
    (bmx["BMXBMI"] > upper_bound)
]

print("IQR:", IQR)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Potential outliers:", len(outliers))
print("Percentage:", len(outliers) / bmx["BMXBMI"].notna().sum() * 100)

IQR: 10.099999999999998
Lower bound: 6.450000000000005
Upper bound: 46.849999999999994
Potential outliers: 189
Percentage: 2.23114154173061


## 6. Systolic Blood Pressure Assessment

In [17]:
# بررسی آماری سه اندازه‌گیری فشار خون سیستولیک

bp_columns = ["BPXOSY1", "BPXOSY2", "BPXOSY3"]

for col in bp_columns:
    print(f"\n{col}")
    print("-" * 40)

    print("Missing:", bpxo[col].isna().sum())
    print(bpxo[col].describe())

    print("Minimum:", bpxo[col].min())
    print("Maximum:", bpxo[col].max())


BPXOSY1
----------------------------------------
Missing: 284
count    7517.000000
mean      119.288546
std        18.561052
min        61.000000
25%       106.000000
50%       117.000000
75%       130.000000
max       232.000000
Name: BPXOSY1, dtype: float64
Minimum: 61.0
Maximum: 232.0

BPXOSY2
----------------------------------------
Missing: 296
count    7505.000000
mean      119.079680
std        18.571271
min        59.000000
25%       106.000000
50%       116.000000
75%       129.000000
max       233.000000
Name: BPXOSY2, dtype: float64
Minimum: 59.0
Maximum: 233.0

BPXOSY3
----------------------------------------
Missing: 321
count    7480.000000
mean      118.917647
std        18.495220
min        50.000000
25%       106.000000
50%       116.000000
75%       129.000000
max       232.000000
Name: BPXOSY3, dtype: float64
Minimum: 50.0
Maximum: 232.0


In [18]:
for col in bp_columns:
    Q1 = bpxo[col].quantile(0.25)
    Q3 = bpxo[col].quantile(0.75)

    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = bpxo[
        (bpxo[col] < lower_bound) |
        (bpxo[col] > upper_bound)
    ]

    valid_count = bpxo[col].notna().sum()

    print(f"\n{col}")
    print("-" * 40)
    print("IQR:", IQR)
    print("Lower bound:", lower_bound)
    print("Upper bound:", upper_bound)
    print("Potential outliers:", len(outliers))
    print("Percentage:", round(len(outliers) / valid_count * 100, 2), "%")


BPXOSY1
----------------------------------------
IQR: 24.0
Lower bound: 70.0
Upper bound: 166.0
Potential outliers: 143
Percentage: 1.9 %

BPXOSY2
----------------------------------------
IQR: 23.0
Lower bound: 71.5
Upper bound: 163.5
Potential outliers: 185
Percentage: 2.47 %

BPXOSY3
----------------------------------------
IQR: 23.0
Lower bound: 71.5
Upper bound: 163.5
Potential outliers: 178
Percentage: 2.38 %


In [19]:
# ایجاد نسخه پاک‌سازی‌شده از داده‌های فشار خون
bpxo_clean = bpxo.copy()

# محاسبه میانگین اندازه‌گیری‌های معتبر هر فرد
bpxo_clean["Mean_Systolic_BP"] = (
    bpxo_clean[bp_columns].mean(axis=1)
)

# شمارش تعداد اندازه‌گیری‌های معتبر هر فرد
bpxo_clean["Valid_Systolic_Readings"] = (
    bpxo_clean[bp_columns].notna().sum(axis=1)
)

# بررسی نتیجه
print("Mean Systolic BP statistics:")
print(bpxo_clean["Mean_Systolic_BP"].describe())

print("\nNumber of valid readings per participant:")
print(
    bpxo_clean["Valid_Systolic_Readings"]
    .value_counts()
    .sort_index()
)

print("\nMissing mean BP:",
      bpxo_clean["Mean_Systolic_BP"].isna().sum())

Mean Systolic BP statistics:
count    7518.000000
mean      119.094418
std        18.151729
min        70.000000
25%       106.333333
50%       116.333333
75%       129.000000
max       232.333333
Name: Mean_Systolic_BP, dtype: float64

Number of valid readings per participant:
Valid_Systolic_Readings
0     283
1      12
2      28
3    7478
Name: count, dtype: int64

Missing mean BP: 283


## 7. Data Merging and Matching

In [20]:
# ادغام فایل‌های دموگرافیک و BMI
merged = demo_clean.merge(
    bmx[["SEQN", "BMXBMI"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# افزودن میانگین فشار خون سیستولیک
merged = merged.merge(
    bpxo_clean[["SEQN", "Mean_Systolic_BP",
                "Valid_Systolic_Readings"]],
    on="SEQN",
    how="left",
    validate="one_to_one"
)

# بررسی نتیجه
print("Merged dataset shape:", merged.shape)
print("Unique participants:", merged["SEQN"].nunique())

print("\nMissing values:")
print(
    merged[
        ["RIDAGEYR", "BMXBMI", "Mean_Systolic_BP"]
    ].isna().sum()
)

Merged dataset shape: (11933, 30)
Unique participants: 11933

Missing values:
RIDAGEYR             189
BMXBMI              3462
Mean_Systolic_BP    4415
dtype: int64


In [21]:
# تفکیک افراد بر اساس وضعیت معاینه
examined = demo_clean[demo_clean["RIDSTATR"] == 2]

# تعداد افراد معاینه‌شده بر اساس سن
print("Examined participants:", len(examined))
print("Under 8:", (examined["RIDAGEYR"] < 8).sum())
print("Age 8 and older:", (examined["RIDAGEYR"] >= 8).sum())
print("Missing age:", examined["RIDAGEYR"].isna().sum())

# بررسی تطبیق شناسه‌ها با فایل فشار خون
print("\nBPXO matching:")
print("Matched with BPXO:",
      examined["SEQN"].isin(bpxo["SEQN"]).sum())

print("Not matched with BPXO:",
      (~examined["SEQN"].isin(bpxo["SEQN"])).sum())

# بررسی تطبیق شناسه‌ها با فایل BMI
print("\nBMX matching:")
print("Matched with BMX:",
      examined["SEQN"].isin(bmx["SEQN"]).sum())

print("Not matched with BMX:",
      (~examined["SEQN"].isin(bmx["SEQN"])).sum())

Examined participants: 8860
Under 8: 926
Age 8 and older: 7801
Missing age: 133

BPXO matching:
Matched with BPXO: 7801
Not matched with BPXO: 1059

BMX matching:
Matched with BMX: 8860
Not matched with BMX: 0


## 8. Analytical Sample Selection

In [22]:
# مرحله اول: انتخاب افراد 20 سال به بالا
adult_data = merged[
    merged["RIDAGEYR"] >= 20
].copy()

print("Adults aged 20 or older:", len(adult_data))

# مرحله دوم: بررسی وجود BMI
with_bmi = adult_data[
    adult_data["BMXBMI"].notna()
].copy()

print("Adults with valid BMI:", len(with_bmi))

# مرحله سوم: بررسی وجود فشار خون
with_bp = with_bmi[
    with_bmi["Mean_Systolic_BP"].notna()
].copy()

print("Adults with valid BMI and BP:", len(with_bp))

# تعداد افراد حذف‌شده در هر مرحله
print("\nExcluded participants:")
print("Missing BMI:", len(adult_data) - len(with_bmi))
print("Missing BP among those with BMI:",
      len(with_bmi) - len(with_bp))

Adults aged 20 or older: 7809
Adults with valid BMI: 5970
Adults with valid BMI and BP: 5805

Excluded participants:
Missing BMI: 1839
Missing BP among those with BMI: 165


## 9. Survey Weights and Sampling Design

In [23]:
print("Weight variable:")
print(merged["WTMEC2YR"].describe())

print("\nMissing weights:")
print(merged["WTMEC2YR"].isna().sum())

print("\nSurvey design variables:")
print("SDMVSTRA missing:", merged["SDMVSTRA"].isna().sum())
print("SDMVPSU missing:", merged["SDMVPSU"].isna().sum())

Weight variable:
count    1.193300e+04
mean     2.740414e+04
std      2.796296e+04
min      5.397605e-79
25%      5.397605e-79
50%      2.171785e+04
75%      3.834115e+04
max      2.271083e+05
Name: WTMEC2YR, dtype: float64

Missing weights:
0

Survey design variables:
SDMVSTRA missing: 0
SDMVPSU missing: 0


In [24]:
tiny_value = 5.397605346934028e-79

print("Count of tiny values:", (merged["WTMEC2YR"] == tiny_value).sum())

print("\nRIDSTATR distribution:")
print(merged["RIDSTATR"].value_counts(dropna=False))

print("\nWeight status by examination status:")
print(
    merged.groupby("RIDSTATR")["WTMEC2YR"]
    .agg(["count", "min", "median", "max"])
)

Count of tiny values: 3073

RIDSTATR distribution:
RIDSTATR
2.0    8860
1.0    3073
Name: count, dtype: int64

Weight status by examination status:
          count           min        median           max
RIDSTATR                                                 
1.0        3073  5.397605e-79  5.397605e-79  5.397605e-79
2.0        8860  4.581595e+03  2.911510e+04  2.271083e+05


In [25]:
tiny_value = 5.397605346934028e-79

valid_weight = with_bp[
    (with_bp["WTMEC2YR"].notna()) &
    (with_bp["WTMEC2YR"] > 0) &
    (with_bp["WTMEC2YR"] != tiny_value)
]

print("Final analytical sample:", len(with_bp))
print("Participants with valid weights:", len(valid_weight))
print("Participants without valid weights:", len(with_bp) - len(valid_weight))

Final analytical sample: 5805
Participants with valid weights: 5805
Participants without valid weights: 0


## 10. Descriptive Statistics

In [26]:
print("Unweighted means:")
print(with_bp[["BMXBMI", "Mean_Systolic_BP"]].mean())

print("\nWeighted means:")
for variable in ["BMXBMI", "Mean_Systolic_BP"]:
    weighted_mean = np.average(
        with_bp[variable],
        weights=with_bp["WTMEC2YR"]
    )
    print(f"{variable}: {weighted_mean:.3f}")


Unweighted means:
BMXBMI               29.730164
Mean_Systolic_BP    122.869308
dtype: float64

Weighted means:
BMXBMI: 29.600
Mean_Systolic_BP: 121.117


In [27]:
descriptive_stats = with_bp[
    ["BMXBMI", "Mean_Systolic_BP"]
].describe().T

descriptive_stats["IQR"] = (
    descriptive_stats["75%"] - descriptive_stats["25%"]
)

descriptive_stats

,count,mean,std,min,25%,50%,75%,max,IQR
BMXBMI,5805.0,29.730164,7.149446,11.1,24.700000,28.400000,33.500000,68.9,8.8
Mean_Systolic_BP,5805.0,122.869308,17.989238,70.0,110.666667,120.333333,132.666667,215.0,22.0
